In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from pyspark.sql.types import DateType


In [0]:
df_clientes = spark.table("silver_dev.sql_finbank.tb_clientes_core")
df_productos = spark.table("silver_dev.sql_finbank.tb_productos_cat")
df_sucursales = spark.table("silver_dev.sql_finbank.tb_sucursales_red")
df_mov = spark.table("silver_dev.sql_finbank.tb_mov_financieros")
df_obl = spark.table("silver_dev.sql_finbank.tb_obligaciones")
df_com = spark.table("silver_dev.sql_finbank.tb_comisiones_log")

In [0]:

# LIMPIEZA

df_clean = (
    df_clientes
    .withColumn("id_cliente", col("id_cli").cast("string"))
    .withColumn("nombre_completo", trim(concat_ws(" ", trim(col("nomb_cli")), trim(col("apell_cli")))))
    .withColumn("fec_nac", to_date(col("fec_nac")))
    .withColumn("cod_segmento", upper(trim(col("cod_segmento"))))
    .withColumn("ciudad_res", upper(trim(col("ciudad_res"))))
    .withColumn("depto_res", upper(trim(col("depto_res"))))
    .withColumn("estado_cliente", upper(trim(col("estado_cli"))))
    .withColumn("canal_adquis", upper(trim(col("canal_adquis"))))
    .dropDuplicates(["id_cliente"])
)

# HASH CAMBIOS

cols_change = [
    "nombre_completo",
    "cod_segmento",
    "ciudad_res",
    "depto_res",
    "estado_cliente",
    "canal_adquis"
]

df_new = (
    df_clean
    .withColumn("hash_diff", sha2(concat_ws("||", *cols_change), 256))
    .withColumn("fecha_inicio", current_date())
    .withColumn("fecha_fin", lit(None).cast("date"))
    .withColumn("es_actual", lit(1))
)

table_name = "gold_dev.dim.dim_clientes"
delta_table = DeltaTable.forName(spark, table_name)


# DATA ACTUAL

df_current = (
    delta_table.toDF()
    .withColumn("id_cliente", col("id_cliente").cast("string"))
    .filter(col("es_actual") == 1)
    .select(
        col("id_cliente"),
        col("hash_diff").alias("hash_diff_cur")
    )
)


# DETECTAR CAMBIOS

df_changes = (
    df_new.alias("new")
    .join(df_current.alias("cur"), "id_cliente", "left")
    .filter(
        col("cur.hash_diff_cur").isNull() |
        (col("new.hash_diff") != col("cur.hash_diff_cur"))
    )
)


# CERRAR VERSIONES ANTERIORES

delta_table.alias("t").merge(
    df_changes.alias("s"),
    "t.id_cliente = s.id_cliente AND t.es_actual = 1"
).whenMatchedUpdate(
    set={
        "fecha_fin": current_date() - expr("INTERVAL 1 DAY"),
        "es_actual": lit(0)
    }
).execute()


# GENERAR NUEVAS SK

max_sk = delta_table.toDF().agg(max("id_cliente_sk")).first()[0]
max_sk = max_sk if max_sk else 0

window_sk = Window.orderBy("id_cliente")

df_insert = (
    df_changes
    .withColumn("id_cliente_sk", row_number().over(window_sk) + max_sk)
    .select(
        col("id_cliente_sk").cast("int"),
        col("id_cliente").cast("string"),
        "nombre_completo",
        "cod_segmento",
        "ciudad_res",
        "depto_res",
        "estado_cliente",
        "canal_adquis",
        "fecha_inicio",
        "fecha_fin",
        "es_actual",
        "hash_diff"
    )
)


# INSERT NUEVAS VERSIONES

df_insert.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(table_name)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


# LECTURA

df_productos = spark.table("silver_dev.sql_finbank.tb_productos_cat")


# LIMPIEZA (FIX NULL desc_prod)

df_clean = (
    df_productos
    .withColumn("id_producto", col("cod_prod"))
    .withColumn(
        "nombre_producto",
        when(col("desc_prod").isNull() | (trim(col("desc_prod")) == ""),
             col("cod_prod")
        ).otherwise(trim(col("desc_prod")))
    )
    .withColumn("tipo_producto", lower(trim(col("tip_prod"))))
    .withColumn("tasa_ea", col("tasa_ea").cast("double"))
)


# FEATURES

df_features = (
    df_clean
    .withColumn(
        "tasa_mensual",
        when(col("tasa_ea").isNotNull() & (col("tasa_ea") > -1),
            pow((1 + col("tasa_ea")), 1/12) - 1
        )
    )
    .withColumn("tasa_mensual", round(col("tasa_mensual"), 6))
    .withColumn(
        "familia_producto",
        when(col("tipo_producto").rlike("credito"), "CREDITO")
        .when(col("tipo_producto").rlike("ahorro"), "AHORRO")
        .when(col("tipo_producto").rlike("corriente"), "TRANSACCIONAL")
        .otherwise("OTROS")
    )
)


# DEDUP

window_dedup = Window.partitionBy("id_producto").orderBy(col("tasa_ea").desc_nulls_last())

df_dedup = (
    df_features
    .withColumn("rn", row_number().over(window_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)


# SK

window_sk = Window.orderBy("id_producto")

dim_productos = (
    df_dedup
    .withColumn("id_producto_sk", row_number().over(window_sk))
    .select(
        col("id_producto_sk").cast("int"),
        "id_producto",
        col("nombre_producto"),
        col("familia_producto"),
        col("tasa_mensual").cast("double"),
        col("plazo_max_meses").cast("int"),
        col("cuota_min").cast("double"),
        col("comision_admin").cast("double")
    )
)


# WRITE

dim_productos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.dim.dim_productos")

In [0]:


# NORMALIZACIÓN TEXTO

def normalize_text(col_name):
    return upper(trim(
        translate(
            col(col_name),
            "áéíóúÁÉÍÓÚ",
            "aeiouAEIOU"
        )
    ))

df_clean = (
    df_sucursales
    .withColumn("ciudad", normalize_text("ciudad"))
    .withColumn("departamento", normalize_text("depto"))
)


# DEDUPLICACIÓN CONTROLADA

df_unique = (
    df_clean
    .select("ciudad", "departamento")
    .dropDuplicates()
)


# SURROGATE KEY (CORRECTO)

window_sk = Window.orderBy("departamento", "ciudad")

dim_geografia = (
    df_unique
    .withColumn("id_geografia_sk", row_number().over(window_sk))
    
  
    # SELECT FINAL (BI READY)
  
    .select(
        col("id_geografia_sk").cast("int"),
        "departamento",
        "ciudad"
    )
)


# VALIDACIONES

assert dim_geografia.count() > 0, "dim_geografia vacío"

assert dim_geografia.select("departamento", "ciudad").distinct().count() == dim_geografia.count(), \
    "Duplicados en dim_geografia"

dim_geografia.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.dim.dim_geografia")

In [0]:
# ============================
# LIMPIEZA
# ============================
df_clean = (
    df_sucursales
    
    # Normalización base
    .withColumn("tipo_punto_atencion",
        upper(trim(coalesce(col("tip_punto"), lit("DESCONOCIDO"))))
    )
)

# ============================
# CLASIFICACIÓN
# ============================
df_features = (
    df_clean
    
    .withColumn("tipo_punto_lower", lower(col("tipo_punto_atencion")))
    
    .withColumn(
        "tipo_canal",
        when(col("tipo_punto_lower").rlike("digital|app|web"), "DIGITAL")
        .when(col("tipo_punto_lower").rlike("oficina|sucursal|atm|cajero|corresponsal"), "FISICO")
        .otherwise("OTRO")
    )
)

# ============================
# DEDUPLICACIÓN
# ============================
df_unique = (
    df_features
    .select("tipo_punto_atencion", "tipo_canal")
    .dropDuplicates()
)

# ============================
# SURROGATE KEY
# ============================
window_sk = Window.orderBy("tipo_punto_atencion")

dim_canal = (
    df_unique
    .withColumn("id_canal_sk", row_number().over(window_sk))
    
    # ============================
    # SELECT FINAL (BI READY)
    # ============================
    .select(
        col("id_canal_sk").cast("int"),
        "tipo_punto_atencion",
        "tipo_canal"
    )
)

# ============================
# VALIDACIONES
# ============================
assert dim_canal.count() > 0, "dim_canal vacío"

assert dim_canal.select("tipo_punto_atencion").distinct().count() == dim_canal.count(), \
    "Duplicados en dim_canal"

dim_canal.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.dim.dim_canal")

In [0]:
# ============================
# RANGO DE FECHAS
# ============================
start_date = "2015-01-01"
end_date = "2030-12-31"

df_dates = spark.sql(f"""
SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) AS fecha_array
""").select(explode(col("fecha_array")).alias("fecha"))

# ============================
# DIMENSION TIEMPO
# ============================
dim_tiempo = (
    df_dates
    
    # Surrogate key correcta
    .withColumn("id_tiempo", date_format(col("fecha"), "yyyyMMdd").cast("int"))
    
    # Atributos
    .withColumn("año", year(col("fecha")))
    .withColumn("mes", month(col("fecha")))
    .withColumn("dia", dayofmonth(col("fecha")))
    .withColumn("trimestre", quarter(col("fecha")))
    
    .withColumn("nombre_mes", date_format(col("fecha"), "MMMM"))
    .withColumn("dia_semana", date_format(col("fecha"), "EEEE"))
    .withColumn("numero_dia_semana", dayofweek(col("fecha")))
    
    .withColumn(
        "es_fin_semana",
        when(dayofweek(col("fecha")).isin([1,7]), 1).otherwise(0)
    )
)

# ============================
# VALIDACIÓN
# ============================
assert dim_tiempo.count() > 0, "dim_tiempo vacío"

dim_tiempo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.dim.dim_tiempo")




In [0]:
# ============================
# NORMALIZACIÓN BASE
# ============================
df_mov_clean = (
    df_mov
    .withColumn("id_cli", col("id_cli").cast("string"))
    .withColumn("cod_prod", upper(trim(col("cod_prod"))))
    .withColumn("cod_canal", upper(trim(col("cod_canal"))))
    .withColumn("fecha", to_date(col("fec_mov")))
    .withColumn("hora_ts", to_timestamp(col("hra_mov"), "HH:mm:ss"))
)

# ============================
# DIMENSIONES
# ============================
dim_clientes = spark.table("gold_dev.dim.dim_clientes")
dim_clientes_current = dim_clientes.filter(col("es_actual") == 1)

dim_productos = dim_productos.withColumn("id_producto", upper(trim(col("id_producto"))))
dim_canal = dim_canal.withColumn("tipo_punto_atencion", upper(trim(col("tipo_punto_atencion"))))

# ============================
# JOINS
# ============================
df_join = (
    df_mov_clean.alias("mov")
    
    .join(
        dim_clientes_current.alias("cli"),
        col("mov.id_cli") == col("cli.id_cliente"),
        "inner"
    )
    
    .join(
        dim_productos.alias("pro"),
        col("mov.cod_prod") == col("pro.id_producto"),
        "left"
    )
    
    .join(
        dim_canal.alias("can"),
        col("mov.cod_canal") == col("can.tipo_punto_atencion"),
        "left"
    )
    
    .join(
        dim_tiempo.alias("ti"),
        col("mov.fecha") == col("ti.fecha"),
        "inner"
    )
)

# ============================
# VENTANAS
# ============================
window_cliente = Window.partitionBy("cli.id_cliente")

window_30d = (
    Window.partitionBy("cli.id_cliente")
    .orderBy(col("mov.fecha").cast("long"))
    .rangeBetween(-30 * 86400, 0)
)

# ============================
# FACT
# ============================
fact_transacciones = (
    df_join
    
    # Métricas
    .withColumn("monto", round(col("mov.vr_mov"), 4))
    .withColumn("monto_abs", round(abs(col("mov.vr_mov")), 4))
    
    # Flags
    .withColumn(
        "horario_habil",
        when((hour(col("hora_ts")) >= 8) & (hour(col("hora_ts")) <= 18), 1).otherwise(0)
    )
    
    .withColumn(
        "transaccion_nocturna",
        when((hour(col("hora_ts")) < 6) | (hour(col("hora_ts")) > 22), 1).otherwise(0)
    )
    
    # Estadísticas cliente
    .withColumn("promedio_cliente", round(avg("mov.vr_mov").over(window_cliente), 4))
    .withColumn("std_cliente", round(stddev("mov.vr_mov").over(window_cliente), 4))
    
    # Rolling 30 días
    .withColumn("monto_30d", round(sum("mov.vr_mov").over(window_30d), 4))
    
    # Z-score
    .withColumn(
    "z_score",
    round(
        when(
            col("std_cliente") != 0,
            (col("mov.vr_mov") - col("promedio_cliente")) / col("std_cliente")
        ).otherwise(0),
        4
        )
    )
    
    # Flag anomalía
    .withColumn("flag_anomalia", when(col("z_score") > 3, 1).otherwise(0))
    
    # Fix null SKs
    .withColumn("id_producto_sk", coalesce(col("pro.id_producto_sk"), lit(-1)))
    .withColumn("id_canal_sk", coalesce(col("can.id_canal_sk"), lit(-1)))
    
    # Select final
    .select(
        col("mov.id_mov").cast("string").alias("id_transaccion"),
        col("cli.id_cliente_sk"),
        col("id_producto_sk"),
        col("id_canal_sk"),
        col("ti.id_tiempo"),
        col("mov.fecha"),
        col("monto"),
        col("monto_abs"),
        col("mov.tip_mov"),
        col("mov.id_dispositivo"),
        col("horario_habil"),
        col("transaccion_nocturna"),
        col("promedio_cliente"),
        col("std_cliente"),
        col("monto_30d"),
        col("z_score"),
        col("flag_anomalia")
    )
)

# ============================
# WRITE
# ============================
fact_transacciones.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.fact.fact_transacciones")

In [0]:
from pyspark.sql.functions import *

# ============================
# NORMALIZACIÓN
# ============================
df_obl_clean = (
    df_obl
    .withColumn("id_cli", col("id_cli").cast("string"))
    .withColumn("dias_mora_act", coalesce(col("dias_mora_act"), lit(0)))
    .withColumn("sdo_capital", coalesce(col("sdo_capital"), lit(0)))
    .withColumn("fecha", to_date(col("fec_venc")))
)

# ============================
# DIM CLIENTES ACTUAL
# ============================
dim_clientes = spark.table("gold_dev.dim.dim_clientes")
dim_clientes_current = dim_clientes.filter(col("es_actual") == 1)

# ============================
# JOINS
# ============================
df_join = (
    df_obl_clean.alias("obl")
    
    .join(
        dim_clientes_current.alias("cli"),
        col("obl.id_cli") == col("cli.id_cliente"),
        "inner"
    )
    
    .join(
        dim_tiempo.alias("ti"),
        col("obl.fecha") == col("ti.fecha"),
        "inner"
    )
)

# ============================
# FACT
# ============================
fact_cartera = (
    df_join
    
    # Bucket mora
    .withColumn(
        "bucket_mora",
        when(col("obl.dias_mora_act") == 0, "AL_DIA")
        .when(col("obl.dias_mora_act") <= 30, "1-30")
        .when(col("obl.dias_mora_act") <= 60, "31-60")
        .when(col("obl.dias_mora_act") <= 90, "61-90")
        .otherwise(">90")
    )
    
    # Clasificación
    .withColumn(
        "clasif_regulatoria",
        when(col("obl.dias_mora_act") <= 30, "A")
        .when(col("obl.dias_mora_act") <= 60, "B")
        .when(col("obl.dias_mora_act") <= 90, "C")
        .when(col("obl.dias_mora_act") <= 120, "D")
        .otherwise("E")
    )
    
    # Provisión
    .withColumn(
        "provision",
        round(
            when(col("clasif_regulatoria") == "A", col("obl.sdo_capital") * 0.01)
            .when(col("clasif_regulatoria") == "B", col("obl.sdo_capital") * 0.05)
            .when(col("clasif_regulatoria") == "C", col("obl.sdo_capital") * 0.20)
            .when(col("clasif_regulatoria") == "D", col("obl.sdo_capital") * 0.50)
            .otherwise(col("obl.sdo_capital") * 1.00),
            4
        )
    )
    
    # Flags
    .withColumn("flag_mora", when(col("obl.dias_mora_act") > 0, 1).otherwise(0))
    
    .withColumn(
        "flag_alto_riesgo",
        when(col("clasif_regulatoria").isin("D", "E"), 1).otherwise(0)
    )
    
    .withColumn(
        "ratio_provision",
        round(
            when(col("obl.sdo_capital") != 0,
                 col("provision") / col("obl.sdo_capital"))
            .otherwise(0),
            4
        )
    )
    
    # SELECT FINAL
    .select(
        col("obl.id_oblig").cast("string").alias("id_obligacion"),
        col("cli.id_cliente_sk"),
        col("ti.id_tiempo"),
        round(col("obl.sdo_capital"), 4).alias("sdo_capital"),
        col("obl.dias_mora_act"),
        col("bucket_mora"),
        col("clasif_regulatoria"),
        col("provision"),
        col("flag_mora"),
        col("flag_alto_riesgo"),
        col("ratio_provision")
    )
)

# ============================
# WRITE
# ============================
fact_cartera.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.fact.fact_cartera")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ============================
# LECTURA
# ============================
df_mov = spark.table("silver_dev.sql_finbank.tb_mov_financieros")
df_com = spark.table("silver_dev.sql_finbank.tb_comisiones_log")

# ============================
# INTERESES
# ============================
mov = (
    df_mov
    .withColumn("id_cli", col("id_cli").cast("string"))
    .withColumn("fecha", to_date(col("fec_mov")))
    .filter(col("fecha").isNotNull())
    .withColumn("periodo", date_format(col("fecha"), "yyyy-MM"))
    .groupBy("id_cli", "periodo")
    .agg(round(sum("vr_mov"), 4).alias("ingresos_intereses"))
)

# ============================
# COMISIONES
# ============================
com = (
    df_com
    .withColumn("id_cli", col("id_cli").cast("string"))
    .withColumn("fecha", to_date(col("fec_cobro")))
    .filter(col("fecha").isNotNull())
    .filter(col("estado_cobro") == "COBRADO")
    .withColumn("periodo", date_format(col("fecha"), "yyyy-MM"))
    .groupBy("id_cli", "periodo")
    .agg(round(sum("vr_comision"), 4).alias("ingresos_comisiones"))
)

# ============================
# UNION
# ============================
rentabilidad = (
    mov.join(com, ["id_cli", "periodo"], "left")
    .fillna({"ingresos_comisiones": 0})
    .withColumn(
        "cltv_mes",
        round(col("ingresos_intereses") + col("ingresos_comisiones"), 4)
    )
)

# ============================
# CLTV (ROLLING 12 MESES)
# ============================
rentabilidad = (
    rentabilidad
    .withColumn("año", substring("periodo", 1, 4).cast("int"))
    .withColumn("mes", substring("periodo", 6, 2).cast("int"))
    .withColumn("periodo_num", col("año") * 12 + col("mes"))
)

window_12m = (
    Window.partitionBy("id_cli")
    .orderBy("periodo_num")
    .rangeBetween(-11, 0)
)

rentabilidad = (
    rentabilidad
    .withColumn(
        "cltv_12m",
        round(sum("cltv_mes").over(window_12m), 4)
    )
)

# ============================
# DIM CLIENTES
# ============================
dim_clientes = spark.table("gold_dev.dim.dim_clientes")
dim_clientes_current = dim_clientes.filter(col("es_actual") == 1)

# ============================
# FACT FINAL
# ============================
fact_rentabilidad_cliente = (
    rentabilidad.alias("rent")
    
    .join(
        dim_clientes_current.alias("cli"),
        col("rent.id_cli") == col("cli.id_cliente"),
        "inner"
    )
    
    .withColumn(
        "id_tiempo",
        regexp_replace(col("periodo"), "-", "").cast("int")
    )
    
    .select(
        col("cli.id_cliente_sk"),
        col("id_tiempo"),
        col("rent.periodo"),
        col("rent.ingresos_intereses"),
        col("rent.ingresos_comisiones"),
        col("rent.cltv_mes"),
        col("rent.cltv_12m")
    )
)

# ============================
# WRITE
# ============================
fact_rentabilidad_cliente.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.fact.fact_rentabilidad_cliente")

In [0]:
from pyspark.sql.functions import *

# ============================
# LECTURA
# ============================
df_obl = spark.table("silver_dev.sql_finbank.tb_obligaciones")

# ============================
# LIMPIEZA
# ============================
df_obl_clean = (
    df_obl
    .withColumn("id_cli", col("id_cli").cast("string"))
    .withColumn("sdo_capital", coalesce(col("sdo_capital"), lit(0)))
    .withColumn("dias_mora_act", coalesce(col("dias_mora_act"), lit(0)))
    .withColumn("fecha", to_date(col("fec_venc")))
    .filter(col("fecha").isNotNull())
)

# ============================
# DIMENSIONES
# ============================
dim_clientes = spark.table("gold_dev.dim.dim_clientes") \
    .filter(col("es_actual") == 1)

# ============================
# JOIN
# ============================
df_join = (
    df_obl_clean.alias("obl")
    .join(
        dim_clientes.alias("cli"),
        col("obl.id_cli") == col("cli.id_cliente"),
        "inner"
    )
)

# ============================
# KPI CARTERA (NIVEL EMPRESARIAL)
# ============================
kpi_cartera = (
    df_join
    
    # 🔥 MENOS GRANULAR (ANALÍTICO)
    .groupBy(
        col("obl.fecha"),
        col("cli.cod_segmento"),
        col("cli.ciudad_res")
    )
    .agg(
        count("*").alias("total_obligaciones"),
        
        round(sum("obl.sdo_capital"), 4).alias("monto_total"),
        
        round(
            sum(
                when(col("obl.dias_mora_act") > 0, col("obl.sdo_capital"))
                .otherwise(0)
            ),
            4
        ).alias("monto_mora"),
        
        countDistinct("cli.id_cliente").alias("clientes_totales"),
        
        countDistinct(
            when(col("obl.dias_mora_act") > 0, col("cli.id_cliente"))
        ).alias("clientes_mora")
    )
    
    # ============================
    # MÉTRICAS KPI
    # ============================
    .withColumn(
        "tasa_mora",
        round(
            when(col("monto_total") > 0,
                 col("monto_mora") / col("monto_total"))
            .otherwise(0),
            4
        )
    )
    
    .withColumn(
        "tasa_clientes_mora",
        round(
            when(col("clientes_totales") > 0,
                 col("clientes_mora") / col("clientes_totales"))
            .otherwise(0),
            4
        )
    )
    
    .withColumn(
        "ticket_promedio",
        round(
            when(col("total_obligaciones") > 0,
                 col("monto_total") / col("total_obligaciones"))
            .otherwise(0),
            4
        )
    )
    
    # ============================
    # SELECT FINAL
    # ============================
    .select(
        col("fecha"),
        col("cod_segmento"),
        col("ciudad_res"),
        col("total_obligaciones").cast("int"),
        col("clientes_totales").cast("int"),
        col("clientes_mora").cast("int"),
        col("monto_total"),
        col("monto_mora"),
        col("tasa_mora"),
        col("tasa_clientes_mora"),
        col("ticket_promedio")
    )
)

# ============================
# WRITE
# ============================
kpi_cartera.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.kpis.kpi_cartera")

In [0]:
from pyspark.sql.functions import *

# ============================
# KPI CARTERA
# ============================
cartera_kpi = (
    fact_cartera
    .agg(
        round(sum("sdo_capital"), 4).alias("cartera_total"),
        round(
            sum(
                when(col("flag_mora") == 1, col("sdo_capital"))
                .otherwise(0)
            ),
            4
        ).alias("monto_mora"),
        countDistinct("id_cliente_sk").alias("clientes_totales"),
        countDistinct(
            when(col("flag_mora") == 1, col("id_cliente_sk"))
        ).alias("clientes_mora")
    )
)

# ============================
# KPI TRANSACCIONES
# ============================
tx_kpi = (
    fact_transacciones
    .agg(
        count("*").alias("total_transacciones"),
        round(sum("monto"), 4).alias("volumen_transacciones"),
        sum(coalesce(col("flag_anomalia"), lit(0))).alias("tx_sospechosas")
    )
)

# ============================
# KPI RENTABILIDAD (CORREGIDO)
# ============================
max_periodo = fact_rentabilidad_cliente.agg(max("id_tiempo")).first()[0]

rent_kpi = (
    fact_rentabilidad_cliente
    .filter(col("id_tiempo") == max_periodo)
    .withColumn(
        "ingreso_total",
        col("ingresos_intereses") + col("ingresos_comisiones")
    )
    .agg(
        round(avg("ingreso_total"), 4).alias("ingreso_promedio_cliente")
    )
)

# ============================
# KPI EJECUTIVO FINAL
# ============================
kpi_ejecutivo = (
    cartera_kpi
    .crossJoin(tx_kpi)
    .crossJoin(rent_kpi)
    
    .withColumn(
        "tasa_mora",
        round(
            when(col("cartera_total") > 0,
                 col("monto_mora") / col("cartera_total"))
            .otherwise(0),
            4
        )
    )
    
    .withColumn(
        "tasa_clientes_mora",
        round(
            when(col("clientes_totales") > 0,
                 col("clientes_mora") / col("clientes_totales"))
            .otherwise(0),
            4
        )
    )
    
    .withColumn(
        "ticket_promedio_tx",
        round(
            when(col("total_transacciones") > 0,
                 col("volumen_transacciones") / col("total_transacciones"))
            .otherwise(0),
            2
        )
    )
    
    .withColumn(
        "ratio_tx_sospechosas",
        round(
            when(col("total_transacciones") > 0,
                 col("tx_sospechosas") / col("total_transacciones"))
            .otherwise(0),
            4
        )
    )
    
    .withColumn("fecha_kpi", current_date())
    
    .select(
        "cartera_total",
        "monto_mora",
        col("clientes_totales").cast("int"),
        col("clientes_mora").cast("int"),
        col("total_transacciones").cast("int"),
        "volumen_transacciones",
        col("tx_sospechosas").cast("int"),
        "ingreso_promedio_cliente",
        "tasa_mora",
        "tasa_clientes_mora",
        "ticket_promedio_tx",
        "ratio_tx_sospechosas",
        "fecha_kpi"
    )
)

# ============================
# VALIDACIÓN
# ============================
assert kpi_ejecutivo.count() == 1, "KPI debe tener una sola fila"

# ============================
# WRITE
# ============================
kpi_ejecutivo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.kpis.kpi_ejecutivo")




In [0]:
from pyspark.sql.functions import *

# ============================
# DIM CLIENTES (SCD ACTUAL)
# ============================
dim_clientes_current = spark.table("gold_dev.dim.dim_clientes") \
    .filter(col("es_actual") == 1)

# ============================
# 1. TRANSACCIONES
# ============================
tx_cliente = (
    fact_transacciones
    .groupBy("id_cliente_sk")
    .agg(
        count("*").alias("num_transacciones"),
        round(sum("monto"), 4).alias("volumen_total"),
        sum(coalesce(col("flag_anomalia"), lit(0))).alias("tx_anomalas")
    )
)

# ============================
# 2. CARTERA
# ============================
cartera_cliente = (
    fact_cartera
    .groupBy("id_cliente_sk")
    .agg(
        round(sum("sdo_capital"), 4).alias("saldo_total"),
        round(
            sum(
                when(col("flag_mora") == 1, col("sdo_capital"))
                .otherwise(0)
            ),
            4
        ).alias("saldo_mora"),
        max(coalesce(col("flag_alto_riesgo"), lit(0))).alias("flag_cliente_riesgoso")
    )
)

# ============================
# 3. RENTABILIDAD
# ============================
rent_cliente = (
    fact_rentabilidad_cliente
    .withColumn(
        "ingreso_total",
        col("ingresos_intereses") + col("ingresos_comisiones")
    )
    .groupBy("id_cliente_sk")
    .agg(
        round(sum("ingreso_total"), 4).alias("ingreso_total_cliente")
    )
)

# ============================
# 4. JOIN 360
# ============================
vista_cliente_360 = (
    dim_clientes_current.alias("cli")
    
    .join(tx_cliente, "id_cliente_sk", "left")
    .join(cartera_cliente, "id_cliente_sk", "left")
    .join(rent_cliente, "id_cliente_sk", "left")
    
    .fillna({
        "num_transacciones": 0,
        "volumen_total": 0,
        "tx_anomalas": 0,
        "saldo_total": 0,
        "saldo_mora": 0,
        "flag_cliente_riesgoso": 0,
        "ingreso_total_cliente": 0
    })
    
    # ============================
    # FEATURES
    # ============================
    .withColumn(
        "tasa_mora_cliente",
        round(
            when(col("saldo_total") > 0,
                 col("saldo_mora") / col("saldo_total"))
            .otherwise(0),
            4
        )
    )
    
    .withColumn(
        "ratio_tx_anomalas",
        round(
            when(col("num_transacciones") > 0,
                 col("tx_anomalas") / col("num_transacciones"))
            .otherwise(0),
            4
        )
    )
    
    .withColumn(
        "segmento_valor",
        when(col("ingreso_total_cliente") > 1000000, "ALTO VALOR")
        .when(col("ingreso_total_cliente") > 300000, "MEDIO VALOR")
        .otherwise("BAJO VALOR")
    )
    
    .withColumn(
        "perfil_cliente",
        when(
            (col("flag_cliente_riesgoso") == 1) | (col("ratio_tx_anomalas") > 0.3),
            "RIESGOSO"
        )
        .when(col("ingreso_total_cliente") > 1000000, "RENTABLE")
        .otherwise("ESTANDAR")
    )
)

# ============================
# WRITE
# ============================
vista_cliente_360.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dev.clientes_360.vista_cliente_360")

# ============================
# VALIDACIÓN
# ============================
assert vista_cliente_360.count() > 0, "vista vacía"


